In [44]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, split, explode, row_number, count, year, regexp_replace, desc
from pyspark.sql.window import Window

In [45]:
spark = SparkSession.builder.appName("ProgrammingLanguages").getOrCreate()
spark

In [46]:
# Загрузка постов
raw_posts = spark.read.format("xml").option("rowTag", "row").load("posts_sample.xml")
raw_posts.show(5, truncate=False)

+-----------------+------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+-------------+-----------------------+-----------------------+--------------+---+-----------------------+-----------------------+----------------------+-----------------+-----------------+------------+---------+-----------+------+------------------------------------------------------+-------

In [47]:
# Загрузка языков программирования
languages_df = spark.read.csv("programming-languages.csv", header=True, inferSchema=True)
languages_df.show(5)

+----------+--------------------+
|      name|       wikipedia_url|
+----------+--------------------+
|   A# .NET|https://en.wikipe...|
|A# (Axiom)|https://en.wikipe...|
|A-0 System|https://en.wikipe...|
|        A+|https://en.wikipe...|
|       A++|https://en.wikipe...|
+----------+--------------------+
only showing top 5 rows


In [48]:
# Преобразуем названия языков в нижний регистр для удобства сравнения
languages = languages_df.select(lower(col("name")).alias("lang_name"))
languages.show(5)

+----------+
| lang_name|
+----------+
|   a# .net|
|a# (axiom)|
|a-0 system|
|        a+|
|       a++|
+----------+
only showing top 5 rows


In [49]:
# Фильтруем по датам (2010-2020) и убираем посты без тегов
period = ("2010-01-01", "2020-12-31")
posts_processed = raw_posts \
  .filter(col("_CreationDate").between(*period)) \
  .filter(col("_Tags").isNotNull()) \
  .select(year("_CreationDate").alias("year"), col("_Tags"))

posts_processed.show(5, truncate=False)

+----+--------------------------------------+
|year|_Tags                                 |
+----+--------------------------------------+
|2010|<c++><character-encoding>             |
|2010|<sharepoint><infopath>                |
|2010|<iphone><app-store><in-app-purchase>  |
|2010|<symfony1><schema><doctrine><fixtures>|
|2010|<java>                                |
+----+--------------------------------------+
only showing top 5 rows


In [50]:
# Заменяем "><" на запятую, убираем крайние <> и разбиваем в массив
tags_cleaned = posts_processed.withColumn(
  "tag",
  explode(split(regexp_replace(regexp_replace(col("_Tags"), "^<|>$", ""), "><", ","), ","))
)

# Джоиним посты с таблицей языков
report_df = tags_cleaned.join(languages, tags_cleaned.tag == languages.lang_name).groupBy("year", "lang_name").count()
report_df.show(5)

+----+----------+-----+
|year| lang_name|count|
+----+----------+-----+
|2019|typescript|   17|
|2017|      perl|    6|
|2011|   haskell|    1|
|2012|      bash|   10|
|2018|      glsl|    1|
+----+----------+-----+
only showing top 5 rows


In [51]:
# Ищем топ-10 для каждого года
window_spec = Window.partitionBy("year").orderBy(col("count").desc())

top_10_result = report_df.withColumn("rank", row_number().over(window_spec)) \
  .filter(col("rank") <= 10) \
  .select(col("year").alias("Year"), col("lang_name").alias("Language"), col("count").alias("Count")) \
  .orderBy("Year", desc("Count"))

top_10_result.show(100)

# Сохранение в Parquet
top_10_result.write.mode("overwrite").parquet("top_10_languages_report.parquet")

+----+-----------+-----+
|Year|   Language|Count|
+----+-----------+-----+
|2010|       java|   52|
|2010|        php|   46|
|2010| javascript|   44|
|2010|     python|   26|
|2010|objective-c|   23|
|2010|          c|   20|
|2010|       ruby|   12|
|2010|     delphi|    8|
|2010|applescript|    3|
|2010|          r|    3|
|2011|        php|  102|
|2011|       java|   93|
|2011| javascript|   83|
|2011|     python|   37|
|2011|objective-c|   34|
|2011|          c|   24|
|2011|       ruby|   20|
|2011|       perl|    9|
|2011|     delphi|    8|
|2011|       bash|    7|
|2012|        php|  154|
|2012| javascript|  132|
|2012|       java|  124|
|2012|     python|   69|
|2012|objective-c|   45|
|2012|       ruby|   27|
|2012|          c|   27|
|2012|       bash|   10|
|2012|          r|    9|
|2012|      scala|    6|
|2013|        php|  198|
|2013| javascript|  198|
|2013|       java|  194|
|2013|     python|   90|
|2013|objective-c|   40|
|2013|          c|   36|
|2013|       ruby|   32|
